# 5. Error analysis

**Stage 1, Step 5 - sections 24, 25, 26.**

Aggregate metrics hide the case that matters. A model can look fine overall while being badly wrong for one category - and the manager who owns that category is the one being handed a confident number.

Three questions:

1. **Where is the error concentrated?** By category, region, channel, promotion state.
2. **Is it biased anywhere it matters?** Bias compounds where dispersion averages out.
3. **What does the model actually lean on**, and does that change with horizon?

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from app.services.container import Container
from ml.forecasting.baselines import attach_seasonal_reference
from ml.forecasting.config import load_forecast_config
from ml.forecasting.dataset import HorizonDataset, build_history, build_horizon_dataset
from ml.forecasting.evaluate import (
    PREDICTED,
    revenue_impact,
    segment_errors,
    zero_demand_summary,
)
from ml.forecasting.explain import (
    importance_by_family,
    permutation_importance_by_horizon,
    recent_history_decay,
    top_drivers,
)
from ml.forecasting.sampling import sample_series
from ml.forecasting.split import build_origin_split, slice_fold
from ml.forecasting.train import build_estimator, train_forecaster

repo = Container().data_repository
config = load_forecast_config().smoke()

sample = sample_series(repo, n_series=config.sampling.n_series, seed=config.sampling.seed)
history = build_history(repo, config, sample)
view = repo.as_of(pd.to_datetime(history["date"]).dt.date.max())
dataset = build_horizon_dataset(history, view, config, sample)
dataset = HorizonDataset(
    frame=attach_seasonal_reference(dataset.frame, history),
    feature_names=[*dataset.feature_names, "seasonal_reference"],
    excluded=dataset.excluded,
)
split = build_origin_split(dataset.frame, config)
model = train_forecaster(dataset, build_estimator("lightgbm", seed=42), config, split)

test = slice_fold(dataset.frame, split.test_start, split.test_end)
scored = test.assign(**{PREDICTED: model.predict(test)})

## 1. Zero and intermittent demand first

This decides which metrics can be believed, so it belongs before the metrics rather than after.

In [ ]:
summary = zero_demand_summary(scored)
for key, value in summary.items():
    print(f"{key:24s} {value:>12,.3f}")

print()
test_metrics = model.metrics["test"]
print(f"MAPE excluded {test_metrics.mape_excluded:,} rows with zero actuals "
      f"({test_metrics.mape_excluded / test_metrics.n:.1%} of the test fold)")
print()
print("This is why WMAPE is the headline. MAPE is undefined at zero and unstable")
print("near it, so a MAPE computed over everything would be dominated by the rows")
print("where the denominator is smallest rather than where the money is.")

## 2. Error by segment (section 24)

In [ ]:
segments = segment_errors(scored)

print("Worst segments")
display(segments.head(12).style.format({"wmape": "{:.1%}", "bias_pct": "{:+.1%}"}))

print("\nBest segments")
display(segments.tail(8).style.format({"wmape": "{:.1%}", "bias_pct": "{:+.1%}"}))

In [ ]:
overall = model.metrics["test"].wmape

for dimension in ("category", "region", "channel"):
    block = segments[segments["segment"] == dimension]
    if block.empty:
        continue
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.barh(block["value"], block["wmape"])
    ax.axvline(overall, color="red", linestyle="--", label=f"overall {overall:.1%}")
    ax.set_title(f"WMAPE by {dimension}")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Bias, which matters more than dispersion (section 25)

Random error averages out over a planning period. A consistent skew does not - it compounds into every inventory decision built on the forecast, always in the same direction.

In [ ]:
biased = segments.reindex(segments["bias_pct"].abs().sort_values(ascending=False).index)

print("Most biased segments")
display(biased.head(10).style.format({"wmape": "{:.1%}", "bias_pct": "{:+.1%}"}))

fig, ax = plt.subplots(figsize=(10, 4))
top = biased.head(12).iloc[::-1]
colours = np.where(top["bias_pct"] >= 0, "tab:orange", "tab:blue")
ax.barh([f"{r.segment}={r.value}" for r in top.itertuples()], top["bias_pct"], color=colours)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("bias (orange = over-forecast)")
ax.set_title("Where the forecast is systematically wrong")
plt.tight_layout()
plt.show()

**A caution before acting on this.** Section 25 says not to correct bias without investigating the cause. One cause is known and structural: rows whose target fell on a stockout are excluded from training, and stockouts are *endogenous* - they happen because demand spiked. Dropping them removes part of the high-demand tail, which biases the model low.

Applying a flat correction factor would paper over that with a number nobody can justify. The honest response is to state the mechanism and its expected direction.

## 4. Revenue impact (section 11)

In [ ]:
impact = revenue_impact(scored)
for key, value in impact.items():
    if "pct" in key:
        print(f"{key:26s} {value:>+12.2%}")
    else:
        print(f"{key:26s} {value:>12,.0f}")

print()
print("Priced at the PLANNED price attached to the target date - the price that")
print("was knowable at forecast time. Section 11 forbids using the realised price:")
print("a forecast made in June cannot be judged against a September price nobody")
print("knew in June, and doing so mixes pricing surprise into a demand metric.")

## 5. What the model leans on, and how that shifts with horizon (section 17)

In [ ]:
importance = permutation_importance_by_horizon(model, scored, config, top_n=25, repeats=2)
families = importance_by_family(importance)

display(families.style.format("{:.4f}"))

fig, ax = plt.subplots(figsize=(11, 5))
families.T.plot(kind="bar", stacked=True, ax=ax)
ax.set_ylabel("summed WMAPE degradation when shuffled")
ax.set_title("What the model relies on, by horizon bucket")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
decay = recent_history_decay(importance)
display(decay.style.format({
    "recent_history_share": "{:.1%}",
    "seasonal_anchor_share": "{:.1%}",
}))

print("Recent history is expected to FALL as the horizon grows: yesterday's sales")
print("say a lot about tomorrow and little about three months out.")
print()
print("The seasonal anchor is expected to hold or RISE. Units from 364 days before")
print("the target are exactly as knowable at h=90 as at h=1, so as the recent")
print("signal decays the anchor's share necessarily grows.")
print()
print("These two are reported separately for that reason. An earlier version")
print("grouped them into one 'demand history' family; the combined share rose")
print("with horizon and looked exactly like a leakage alarm, while being nothing")
print("of the sort. Two features with opposite horizon profiles must not be")
print("averaged into a single diagnostic.")

In [ ]:
print("Top drivers at the shortest horizon")
display(top_drivers(importance, bucket="h1-3", n=8))

print("Top drivers at the longest horizon")
display(top_drivers(importance, bucket="h57-90", n=8))

**A caveat on reading these numbers at all.** The model sits at roughly 1.25x the irreducible noise floor, which means only about nine percentage points of WMAPE are learnable in the first place. When the total learnable signal is that small, permutation degradations for individual features are correspondingly small and noisy - some come out at or below zero. Treat the family-level pattern as the finding and individual feature ranks as indicative.

**None of this is causal.** Permutation importance measures what the fitted function *relies on*, which is a statement about the model rather than about demand. A feature can rank highly because it proxies something the model cannot see directly.

SHAP is deliberately absent - it is not a project dependency, and at this scale it costs considerably more than it adds for the question being asked. Step 4 made the same call.

---

## Findings

| Question | Answer |
|---|---|
| Which segments are worst? | Reported above; the aggregate hides a real spread |
| Is the error biased? | Yes, and one mechanism is known: excluded stockout targets remove part of the high-demand tail |
| Should the bias be corrected? | Not blindly. Section 25 asks for the cause first, and here the cause is a deliberate modelling choice |
| What drives the forecast? | Recent history at short horizons, giving way to the seasonal anchor, calendar and planned promotion at long ones |

### Next

`06_forecast_validation.ipynb` - the leakage checks, the serving path, and the refusal behaviour.